In [1]:
!pip install timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.6 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 

In [2]:
from torchvision import transforms
train_tf = transforms.Compose([
    transforms.Resize((244, 244)),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.RandomPerspective(
        distortion_scale=0.15,
        p=0.3
    ),
    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1, 0.1)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tf = transforms.Compose([
    transforms.Resize((244, 244)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [3]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

train_ds = ImageFolder(
    "/kaggle/input/datasets/hoangvudinh0301/3-nation-plate/plate_dataset/train",
    transform=train_tf
)
val_ds = ImageFolder(
    "/kaggle/input/datasets/hoangvudinh0301/3-nation-plate/plate_dataset/val",
    transform=val_tf
)
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2
)
val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [4]:
print(train_ds.classes)
print(len(train_ds.classes))
num_classes = len(train_ds.classes)
print(num_classes)

['china', 'lao', 'vietnamese']
3
3


In [5]:
import torch
import timm
import torch.nn as nn
from tqdm import tqdm
num_classes = len(train_ds.classes)

model = timm.create_model(
    'efficientnet_b3',
    pretrained=True,
    num_classes=num_classes
)

device = "cuda"
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

best_acc = 0
for epoch in range(50):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    pbar = tqdm(train_loader)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        train_acc = correct / total
        pbar.set_description(
            f"Epoch {epoch+1}"
        )
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{train_acc:.4f}"})

    scheduler.step()
    train_loss /= len(train_loader)
    train_acc = correct / total
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_loss /= len(val_loader)
    val_acc = correct / total
    print(
        f"\nEpoch [{epoch+1}/50]"
        f"\nTrain Loss: {train_loss:.4f}"
        f"\nTrain Acc : {train_acc:.4f}"
        f"\nVal Loss  : {val_loss:.4f}"
        f"\nVal Acc   : {val_acc:.4f}"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_efficientnet_b0.pth")
        print("Saved Best Model")

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Epoch 1: 100%|██████████| 86/86 [00:46<00:00,  1.84it/s, loss=0.7309, acc=0.8988]



Epoch [1/50]
Train Loss: 0.7372
Train Acc : 0.8988
Val Loss  : 0.4111
Val Acc   : 0.9692
Saved Best Model


Epoch 2: 100%|██████████| 86/86 [00:40<00:00,  2.11it/s, loss=0.3413, acc=0.9839]



Epoch [2/50]
Train Loss: 0.3646
Train Acc : 0.9839
Val Loss  : 0.3378
Val Acc   : 0.9846
Saved Best Model


...


Epoch [48/50]
Train Loss: 0.2969
Train Acc : 0.9996
Val Loss  : 0.3238
Val Acc   : 0.9868


Epoch 49: 100%|██████████| 86/86 [00:44<00:00,  1.94it/s, loss=0.2916, acc=1.0000]



Epoch [49/50]
Train Loss: 0.2922
Train Acc : 1.0000
Val Loss  : 0.3157
Val Acc   : 0.9912


Epoch 50: 100%|██████████| 86/86 [00:51<00:00,  1.68it/s, loss=0.2931, acc=1.0000]



Epoch [50/50]
Train Loss: 0.2920
Train Acc : 1.0000
Val Loss  : 0.3159
Val Acc   : 0.9890


In [14]:
test_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_ds = ImageFolder('/kaggle/input/datasets/hoangvudinh0301/test-ch-la-vn/test_plate_classify', transform=test_tf)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [15]:
import timm
import torch

num_classes = len(test_ds.classes)
model = timm.create_model('efficientnet_b3', pretrained=False, num_classes=num_classes)
model.load_state_dict(torch.load("/kaggle/working/best_efficientnet_b0.pth"))
device = "cuda"
model = model.to(device)
model.eval()

correct = 0
total = 0
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total
print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.9307


In [13]:
import torch
import timm
import os

from PIL import Image
from torchvision import transforms

classes = ["CN","LA","VN"]

model = timm.create_model(
    'efficientnet_b3',
    pretrained=False,
    num_classes=len(classes)
)

model.load_state_dict(
    torch.load("/kaggle/working/best_efficientnet_b0.pth")
)

model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])
test_dir = "/kaggle/input/datasets/hoangvudinh0301/mutil-plate/plate/val/china"
for file_name in os.listdir(test_dir):
    img_path = os.path.join(test_dir, file_name)
    image = Image.open(img_path).convert("RGB")
    x = transform(image).unsqueeze(0)
    x = x.to(device)
    with torch.no_grad():
        outputs = model(x)
        pred = outputs.argmax(1).item()
    country = classes[pred]
    print(f"{file_name} -> {country}")

china_155.png -> CN
china_167.png -> CN
china_58.png -> CN
china_116.png -> CN
china_88.png -> CN
china_35.png -> CN
china_52.png -> CN
china_153.png -> CN
china_31.png -> CN
china_11.png -> CN
china_71.png -> CN
china_195.png -> CN
china_63.png -> CN
china_37.png -> CN
china_157.png -> CN
china_177.png -> CN
china_48.png -> CN
china_164.png -> CN
china_106.png -> CN
china_166.png -> CN
china_40.png -> CN
china_147.png -> CN
china_156.png -> CN
china_184.png -> CN
china_191.png -> CN
china_14.png -> CN
china_180.png -> CN
china_1.png -> CN
china_89.png -> CN
china_154.png -> CN
china_172.png -> CN
china_62.png -> CN
china_182.png -> CN
china_64.png -> CN
china_51.png -> CN
china_81.png -> CN
china_13.png -> CN
china_68.png -> CN
china_93.png -> CN
china_189.png -> CN
china_132.png -> CN
china_87.png -> CN
china_57.png -> CN
china_136.png -> CN
china_130.png -> CN
china_135.png -> CN
china_17.png -> CN
china_126.png -> CN
china_134.png -> CN
china_15.png -> CN
china_9.png -> CN
china_11